# edge-tts 테스트 — schwa deletion 수정 버전

`deva` → 데-와 (기존: 데브), `buddha` → 붓-다 (기존: 붓드)

In [ ]:
!pip install -q edge-tts
print('설치 완료!')

In [ ]:
# 데바나가리 변환 함수 + schwa deletion 방지
import re as _re

CONSONANTS = {
    'kh': '\u0916', 'k': '\u0915', 'gh': '\u0918', 'g': '\u0917', '\u1e45': '\u0919',
    'ch': '\u091b', 'c': '\u091a', 'jh': '\u091d', 'j': '\u091c', '\u00f1': '\u091e',
    '\u1e6dh': '\u0920', '\u1e6d': '\u091f', '\u1e0dh': '\u0922', '\u1e0d': '\u0921', '\u1e47': '\u0923',
    'th': '\u0925', 't': '\u0924', 'dh': '\u0927', 'd': '\u0926', 'n': '\u0928',
    'ph': '\u092b', 'p': '\u092a', 'bh': '\u092d', 'b': '\u092c', 'm': '\u092e',
    'y': '\u092f', 'r': '\u0930', 'l': '\u0932', '\u1e37': '\u0933',
    'v': '\u0935', 's': '\u0938', 'h': '\u0939',
}
VOWELS_IND = { '\u0101': '\u0906', 'a': '\u0905', '\u012b': '\u0908', 'i': '\u0907', '\u016b': '\u090a', 'u': '\u0909', 'e': '\u090f', 'o': '\u0913' }
VOWELS_DEP = { '\u0101': '\u093e', 'a': '', '\u012b': '\u0940', 'i': '\u093f', '\u016b': '\u0942', 'u': '\u0941', 'e': '\u0947', 'o': '\u094b' }
VIRAMA = '\u094d'

def prevent_schwa(deva):
    """힌디어 schwa deletion 방지 — 단어 끝 자음에 ा 추가"""
    def fix_word(w):
        if not w.strip() or w == ',': return w
        code = ord(w[-1])
        if 0x0915 <= code <= 0x0939:
            return w + '\u093e'
        return w
    return ''.join(fix_word(part) for part in _re.split(r'(\s+|,)', deva))

def pali_to_devanagari(roman):
    result = ''; i = 0; s = roman.lower()
    while i < len(s):
        ch = s[i]
        if ch in ' ,.;:!?-\n\r\t/': result += ',' if ch == '/' else ch; i += 1; continue
        if ch == '\u1e43': result += '\u0902'; i += 1; continue
        three = s[i:i+3]; two = s[i:i+2]
        consonant = None; consumed = 0
        if three in CONSONANTS: consonant = CONSONANTS[three]; consumed = 3
        elif two in CONSONANTS: consonant = CONSONANTS[two]; consumed = 2
        elif ch in CONSONANTS: consonant = CONSONANTS[ch]; consumed = 1
        if consonant:
            i += consumed
            if i < len(s) and s[i] in VOWELS_IND: result += consonant + VOWELS_DEP[s[i]]; i += 1
            else: result += consonant + VIRAMA
            continue
        if ch in VOWELS_IND: result += VOWELS_IND[ch]; i += 1; continue
        result += ch; i += 1
    return prevent_schwa(result)

# 테스트
for t in ['deva', 'buddha', 'dhamma', 'bhikkhu', 'nibbāna']:
    print(f'  {t} → {pali_to_devanagari(t)}')
print('변환 함수 준비 (schwa 방지 적용)')

In [ ]:
# 언어별 비교 — 로마자 직접 vs 데바나가리
import edge_tts, os, IPython.display as ipd

os.makedirs('audio_mp3', exist_ok=True)

async def speak(text, voice, rate, filename):
    comm = edge_tts.Communicate(text, voice, rate=rate)
    await comm.save(filename)

words = ['deva', 'buddha', 'dhamma', 'aṅguli', 'bhikkhu', 'nibbāna',
         'Buddhaṃ saraṇaṃ gacchāmi']

voices = [
    ('힌디+데바나가리', 'hi-IN-MadhurNeural', '-30%', True),
    ('이탈리아+로마자', 'it-IT-DiegoNeural', '-30%', False),
    ('영어+로마자', 'en-US-GuyNeural', '-40%', False),
]

for word in words:
    print(f'\n=== {word} ===')
    for label, voice, rate, use_deva in voices:
        text = pali_to_devanagari(word) if use_deva else word.replace('ṃ', 'm').replace('ṅ', 'ng').replace('ṭ', 't').replace('ḍ', 'd').replace('ṇ', 'n').replace('ñ', 'ny').replace('ā', 'aa').replace('ī', 'ee').replace('ū', 'oo').replace('ḷ', 'l')
        fname = f'audio_mp3/{word[:10]}_{label[:4]}.mp3'
        print(f'  {label}: {text}')
        await speak(text, voice, rate, fname)
        ipd.display(ipd.Audio(fname))